# M2.0a Primer: Gauss' Law by Symmetry

Book pointer: chapter 4, sections 4.1 (motivation), 4.2 (integral form), 4.3 (differential form), and the conductor results of section 7.2.

First of two primers for module 2. The M1 primers ended with the flux of a point-charge field being $4\pi$ through any surface around it. This notebook turns that fact into Gauss' law and then into the pen-and-paper method that solves every highly symmetric charge distribution in three lines: choose a surface that matches the symmetry, argue that the field is constant on it, and divide. Every hand result is checked against a brute-force superposition sum built from thousands of point charges, so you can see both that the method works and where it stops working.

## 1. Warm-up

1. State Gauss' law in integral form. What exactly is $Q_{\text{enc}}$, and what happens to the flux if a charge sits just outside the surface?
2. For a point charge, why is the flux through a sphere of radius $R$ independent of $R$?
3. What three properties must a Gaussian surface have for the symmetry method to work?
4. Field inside a uniformly charged spherical shell: what is it, and why?
5. Field just outside the surface of a conductor carrying surface charge $\sigma$.

## 2. The idea in one paragraph

Coulomb's law gives the field of a point charge, and superposition gives the field of anything else as a sum. Gauss' law repackages the same physics: the outward flux of $\mathbf{E}$ through any closed surface equals the enclosed charge over $\epsilon_0$. That is always true, but it is only useful when symmetry tells you the direction of $\mathbf{E}$ and that its magnitude is the same everywhere on a well-chosen surface. Then the flux integral collapses to $|\mathbf{E}|$ times an area and the field follows by division. Three symmetries qualify: spherical, cylindrical and planar. For everything else, Gauss' law still holds but does not give the field, and you go back to superposition, which is what the rest of module 2 does on a grid.

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from emlib import fields
from emlib.fields import EPS0, K

plt.rcParams["figure.figsize"] = (6, 5)
np.set_printoptions(precision=4, suppress=True)

### 3.1 From $4\pi$ to Gauss' law

For a point charge $q$ at the origin, $\mathbf{E} = \dfrac{q}{4\pi\epsilon_0}\dfrac{\hat{\mathbf{r}}}{r^2}$. The flux through a sphere of radius $R$ is $|\mathbf{E}|\times$ area $= \dfrac{q}{4\pi\epsilon_0R^2}\cdot4\pi R^2 = q/\epsilon_0$. Primer 00b showed numerically that the same $4\pi$ comes out for a cube and for off-centre surfaces: the flux counts the charge, not the geometry. With superposition, the flux of a sum of fields is the sum of the fluxes, so for any collection of charges

$$\oint_S \mathbf{E}\cdot d\mathbf{S} = \frac{Q_{\text{enc}}}{\epsilon_0}.$$

Charges outside the surface contribute nothing: their field lines enter on one side and leave on the other. Charges inside contribute $q/\epsilon_0$ each, wherever they sit inside. That is the integral form of Gauss' law, and it is exactly equivalent to Coulomb's law plus superposition.

**Worked example 1.** Three charges, $+2\,$nC and $-1\,$nC inside a closed surface and $+5\,$nC outside. Flux $= (2 - 1)\times10^{-9}/\epsilon_0 = 113\,\mathrm{V\,m}$. Move the $+5\,$nC charge anywhere outside: no change. Move it inside: flux jumps to $6\times10^{-9}/\epsilon_0 = 678\,\mathrm{V\,m}$. The code checks this with the box-flux routine from module 2 applied to the superposition field on a 3D grid.

In [ ]:
from emlib import grids
x, y, z, P = grids.grid3d((-1, 1), (-1, 1), (-1, 1), 61)
h = grids.spacing(x, y, z)
inside = [[0.2, 0.1, 0.0], [-0.3, -0.2, 0.1]]
for third, where in (([0.0, 0.0, 1.6], "outside"), ([0.4, -0.4, 0.3], "inside")):
    E = fields.point_charges_E(inside + [third], [2e-9, -1e-9, 5e-9], P)
    flux = fields.box_flux(E, h, (10, 10, 10), (50, 50, 50))          # the box |x|,|y|,|z| <= 2/3
    enclosed = 1e-9 if where == "outside" else 6e-9
    print(f"+5 nC {where:8s}: flux = {flux:8.2f} V m    Q_enc / eps0 = {enclosed / EPS0:8.2f} V m")

### 3.2 The symmetry method

To get $\mathbf{E}$ out of Gauss' law you need a Gaussian surface on which the integral is trivial. Each part of the surface must satisfy one of two conditions:

1. $\mathbf{E}$ is perpendicular to the surface and $|\mathbf{E}|$ is the same everywhere on it. Then the flux through that part is $|\mathbf{E}|\times$ area.
2. $\mathbf{E}$ is parallel to the surface. Then the flux through that part is zero.

Both conditions come from symmetry arguments, not from calculation. The argument always has the form: "the charge distribution looks the same after some transformation (rotation, translation, reflection), so the field must look the same too, so it cannot have a component in such-and-such direction." Three distributions have enough symmetry: anything spherically symmetric (sphere, shell, concentric shells), anything with infinite cylindrical symmetry (long line, long cylinder, coaxial shells), and anything with infinite planar symmetry (plane, slab, parallel sheets). Each gets one of the three surfaces below, and the rest of this notebook works through them.

### 3.3 Spherical symmetry: a uniformly charged ball

A ball of radius $R$ with total charge $Q$ spread uniformly, so $\rho = 3Q/4\pi R^3$. Symmetry: rotating the ball about any axis through its centre changes nothing, so $\mathbf{E}$ can only point radially, and $|\mathbf{E}|$ can only depend on $r$. Gaussian surface: a concentric sphere of radius $r$. On it, condition 1 holds everywhere: flux $= E(r)\cdot4\pi r^2$.

**Worked example 2 (outside, $r > R$).** $Q_{\text{enc}} = Q$. So $E\cdot4\pi r^2 = Q/\epsilon_0$ and
$$E(r) = \frac{Q}{4\pi\epsilon_0 r^2}, \qquad r \ge R.$$
The same as a point charge $Q$ at the centre. Outside, a uniformly charged ball is indistinguishable from a point charge.

**Worked example 3 (inside, $r < R$).** Only the charge within radius $r$ counts: $Q_{\text{enc}} = \rho\cdot\tfrac43\pi r^3 = Q\,r^3/R^3$. So $E\cdot4\pi r^2 = Qr^3/\epsilon_0R^3$ and
$$E(r) = \frac{Q\,r}{4\pi\epsilon_0 R^3}, \qquad r \le R.$$
The field grows linearly from zero at the centre, reaches $Q/4\pi\epsilon_0R^2$ at the surface, then falls as $1/r^2$. At the centre every direction is equivalent, so the field must vanish there: a symmetry argument gives that result without any calculation.

The check builds the ball from about 36 000 point charges on a cubic lattice. Inside the ball the field of a lattice of charges is grainy: it spikes near each charge and dips between them, and a single sample line gives a noisy answer. The macroscopic field is the average over many nearby points, so at each radius the code averages the radial component over 32 directions. That averaging is also how the smooth $\rho$ of the hand calculation relates to the lumpy reality of charges, a point that returns with polarization in module 3.

In [ ]:
R, Q = 1.0, 1e-9
n = 41
s = -R + (np.arange(n) + 0.5) * (2 * R / n)
Xc, Yc, Zc = np.meshgrid(s, s, s, indexing="ij")
in_ball = Xc**2 + Yc**2 + Zc**2 <= R**2
cells = np.stack([Xc[in_ball], Yc[in_ball], Zc[in_ball]], axis=-1)
q_cell = np.full(len(cells), Q / len(cells))                 # equal charge per cell, total Q
print(f"ball built from {len(cells)} point charges")

def E_ball(r):
    return np.where(r <= R, K * Q * r / R**3, K * Q / r**2)

r_obs = np.linspace(0.05, 3.0, 60)
direction = np.array([0.6, 0.48, 0.64])                      # a unit vector used for single-line checks later on
k32 = np.arange(32) + 0.5
dirs = np.stack([np.sin(np.arccos(1 - 2 * k32 / 32)) * np.cos(np.pi * (1 + 5**0.5) * k32),
                 np.sin(np.arccos(1 - 2 * k32 / 32)) * np.sin(np.pi * (1 + 5**0.5) * k32), 1 - 2 * k32 / 32], axis=-1)   # 32 directions
obs = (r_obs[:, None, None] * dirs[None, :, :]).reshape(-1, 3)
E_num = fields.point_charges_E(cells, q_cell, obs).reshape(len(r_obs), 32, 3)
E_rad_all = np.einsum("rdk,dk->rd", E_num, dirs)             # radial component in each direction
E_rad = E_rad_all.mean(axis=1)                                # averaged over directions: the macroscopic field
E_tan = np.linalg.norm(E_num - E_rad_all[..., None] * dirs[None], axis=-1).mean(axis=1)

plt.plot(r_obs, E_ball(r_obs), "k-", lw=3, alpha=0.3, label="Gauss by hand")
plt.plot(r_obs, E_rad, "C1.", label="superposition of 36k charges, direction-averaged")
plt.axvline(R, color="grey", ls=":"); plt.xlabel("r / R"); plt.ylabel("E_r  [V/m]"); plt.legend(); plt.title("uniformly charged ball"); plt.show()
print(f"max relative error, r > 1.2R: {np.abs(E_rad / E_ball(r_obs) - 1)[r_obs > 1.2].max():.1e}")
print(f"max |error| inside, as a fraction of the surface field E(R): {(np.abs(E_rad - E_ball(r_obs)) / E_ball(R))[r_obs < R].max():.1e}   (the field vanishes at the centre, so a relative error there is meaningless)")
print(f"mean tangential component / radial, r > 1.2R: {(E_tan / np.abs(E_rad))[r_obs > 1.2].max():.1e}   (symmetry says zero)")

### 3.4 A spherical shell, and what "enclosed" buys you

A thin shell of radius $R$ carrying charge $Q$. Same symmetry, same Gaussian sphere. Outside: $Q_{\text{enc}} = Q$, so $E = Q/4\pi\epsilon_0r^2$, a point charge again. Inside: $Q_{\text{enc}} = 0$, so

$$E = 0 \quad\text{everywhere inside the shell.}$$

This is a strong statement. Every piece of the shell pulls on a test charge inside, and the pulls cancel exactly at every interior point, not just the centre. By superposition alone you would have to integrate over the shell to show it. Gauss' law gives it in one line. The check builds the shell from 20 000 patches and evaluates the field at several interior and exterior points.

**Worked example 4.** Two concentric shells, $+Q$ at radius $a$ and $-Q$ at radius $b > a$ (a spherical capacitor). For $r < a$: nothing enclosed, $E = 0$. For $a < r < b$: $Q_{\text{enc}} = Q$, $E = Q/4\pi\epsilon_0r^2$. For $r > b$: $Q_{\text{enc}} = 0$, $E = 0$. The field lives only in the gap.

In [ ]:
def shell_charges(R, Q, N):
    k = np.arange(N) + 0.5
    theta = np.arccos(1 - 2 * k / N); phi = np.pi * (1 + 5**0.5) * k
    pos = R * np.stack([np.sin(theta) * np.cos(phi), np.sin(theta) * np.sin(phi), np.cos(theta)], axis=-1)
    return pos, np.full(N, Q / N)

pos, qs = shell_charges(1.0, Q, 20000)
test_points = np.array([[0, 0, 0], [0.3, 0.2, 0.1], [-0.5, 0.4, 0.3], [0.0, 0.0, 0.85], [1.5, 0.0, 0.0], [0.0, 2.0, 0.0]])
E = fields.point_charges_E(pos, qs, test_points)
for p, e in zip(test_points, E):
    r = np.linalg.norm(p)
    print(f"r = {r:.3f} ({'inside ' if r < 1 else 'outside'}): |E| = {np.linalg.norm(e):10.4f}   Gauss: {0.0 if r < 1 else K * Q / r**2:10.4f}")

# Worked example 4: a spherical capacitor.
pos_a, q_a = shell_charges(0.5, Q, 20000); pos_b, q_b = shell_charges(1.0, -Q, 20000)
r_line = np.array([0.2, 0.4, 0.6, 0.8, 0.95, 1.2, 2.0])
E_cap = fields.point_charges_E(np.vstack([pos_a, pos_b]), np.concatenate([q_a, q_b]), r_line[:, None] * direction)
E_cap_hand = np.where(r_line < 0.5, 0.0, np.where(r_line < 1.0, K * Q / r_line**2, 0.0))
print("\nspherical capacitor, E_r at r =", r_line)
print("  superposition:", np.einsum("ij,j->i", E_cap, direction))
print("  Gauss by hand:", E_cap_hand)

### 3.5 Cylindrical symmetry: a long line and a long cylinder

An infinitely long straight line with charge $\lambda$ per unit length along the $z$ axis. Symmetry: sliding along $z$, rotating about $z$, and reflecting $z \to -z$ all leave the line unchanged. So $\mathbf{E}$ cannot depend on $z$ or $\phi$, cannot have a $\phi$ component (the reflection $\phi \to -\phi$ would flip it), and cannot have a $z$ component (the reflection $z \to -z$ would flip it). It is purely radial, $\mathbf{E} = E(r)\hat{\mathbf{r}}$ in cylindrical coordinates.

Gaussian surface: a coaxial cylinder of radius $r$ and length $L$. On the curved side, condition 1: flux $= E(r)\cdot2\pi rL$. On the two flat ends, condition 2: $\mathbf{E}$ is radial, parallel to the ends, flux zero. Enclosed charge $\lambda L$.

**Worked example 5.** $E\cdot2\pi rL = \lambda L/\epsilon_0$, so
$$E(r) = \frac{\lambda}{2\pi\epsilon_0 r}.$$
The field falls as $1/r$, not $1/r^2$: a line charge extends to infinity, and the far parts keep contributing. The length $L$ cancelled, as it must, since the answer cannot depend on how long we chose to make the imaginary cylinder.

**Worked example 6.** A solid cylinder of radius $R$ with uniform $\rho$ ($\lambda = \rho\pi R^2$). Inside, $Q_{\text{enc}} = \rho\pi r^2L$, so $E = \rho r/2\epsilon_0$, linear in $r$. Outside, $E = \rho R^2/2\epsilon_0 r = \lambda/2\pi\epsilon_0 r$. Same shape as the ball: linear inside, then the point-like (here line-like) fall-off outside.

The check uses a line of finite length $2L_{\text{line}}$ built from points. Close to the line the $1/r$ law holds; as $r$ approaches $L_{\text{line}}$ the finite ends show, and the field drops below the infinite-line value. That failure is informative: Gauss' law with the symmetry argument assumes the line is infinite, and "infinite" in practice means long compared with the distance at which you look.

In [ ]:
lam, L_line, N = 1e-9, 50.0, 20000
zs = -L_line + (np.arange(N) + 0.5) * (2 * L_line / N)
line_pos = np.stack([0 * zs, 0 * zs, zs], axis=-1)
line_q = np.full(N, lam * 2 * L_line / N)
r_vals = np.array([0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 60.0])
E_line = fields.point_charges_E(line_pos, line_q, np.stack([r_vals, 0 * r_vals, 0 * r_vals], axis=-1))[:, 0]
print("      r    superposition    lambda/(2 pi eps0 r)    ratio")
for r, e in zip(r_vals, E_line):
    print(f"{r:7.1f}   {e:12.4f}   {lam / (2 * np.pi * EPS0 * r):18.4f}   {e / (lam / (2 * np.pi * EPS0 * r)):8.4f}")
print(f"the line has half-length {L_line}: the infinite-line formula holds while r << {L_line}")

### 3.6 Planar symmetry: a sheet and a slab

An infinite plane sheet with charge $\sigma$ per unit area in the plane $z = 0$. Symmetry: any translation in $x$ or $y$ and any rotation about $z$ leaves it unchanged, so $\mathbf{E}$ depends on $z$ only and points along $\pm\hat{\mathbf{z}}$. Reflection $z \to -z$ maps the field above to the field below, so $E_z(-z) = -E_z(z)$: it points away from the sheet on both sides (for positive $\sigma$).

Gaussian surface: a pillbox of face area $A$ straddling the sheet, from $-z$ to $+z$. Both faces, condition 1: flux $= E\cdot A$ on each, $2EA$ total. The curved side, condition 2: zero. Enclosed charge $\sigma A$.

**Worked example 7.** $2EA = \sigma A/\epsilon_0$, so
$$E = \frac{\sigma}{2\epsilon_0} \quad\text{on each side, independent of distance.}$$
No fall-off at all. However far you are from an infinite sheet, it looks the same, so the field cannot change. Two parallel sheets with $\pm\sigma$ (a parallel-plate capacitor): between them the two fields add, $E = \sigma/\epsilon_0$; outside they cancel, $E = 0$.

**Worked example 8.** A slab of thickness $2d$ with uniform $\rho$, centred on $z = 0$. Pillbox from $-z$ to $z$ with $|z| < d$: enclosed charge $\rho\cdot2zA$, so $2EA = 2\rho zA/\epsilon_0$ and $E = \rho z/\epsilon_0$, linear in $z$, zero on the midplane. Outside the slab, $E = \rho d/\epsilon_0 = \sigma_{\text{total}}/2\epsilon_0$ with $\sigma_{\text{total}} = 2\rho d$: the slab looks like a sheet from outside.

The check uses a square plate of side $a$ built from a grid of charges. Above the centre, at heights small compared with $a$, the field is $\sigma/2\epsilon_0$. At heights comparable to $a$ the edges matter and it falls off, eventually like a point charge $Q = \sigma a^2$.

In [ ]:
a, sigma, m = 20.0, 1e-9, 200
u = -a / 2 + (np.arange(m) + 0.5) * (a / m)
U, V = np.meshgrid(u, u, indexing="ij")
plate_pos = np.stack([U.ravel(), V.ravel(), np.zeros(m * m)], axis=-1)
plate_q = np.full(m * m, sigma * (a / m) ** 2)
heights = np.array([0.2, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0])
E_plate = fields.point_charges_E(plate_pos, plate_q, np.stack([0 * heights, 0 * heights, heights], axis=-1))[:, 2]
print("   z     superposition   sigma/(2 eps0)   ratio    point charge Q/(4 pi eps0 z^2)")
for zz, e in zip(heights, E_plate):
    print(f"{zz:5.1f}   {e:12.3f}   {sigma / (2 * EPS0):12.3f}   {e / (sigma / (2 * EPS0)):6.3f}   {K * sigma * a**2 / zz**2:12.3f}")
print(f"the plate has side {a}: the infinite-sheet formula holds while z << {a}, and the point-charge formula takes over for z >> {a}")

### 3.7 Conductors: Gauss' law at a surface

Inside a conductor in equilibrium $\mathbf{E} = 0$ (otherwise charges would move). Apply Gauss' law to a small pillbox straddling the conductor's surface: the inner face is inside the metal where $\mathbf{E} = 0$, the outer face sees the field $E_n$ just outside, and the sides contribute nothing because the field just outside is normal to the surface (a tangential component would push surface charges along the surface). So $E_nA = \sigma A/\epsilon_0$ and

$$E_n = \frac{\sigma}{\epsilon_0} \quad\text{just outside a conductor.}$$

Twice the field of an isolated sheet with the same $\sigma$. The difference: the conductor's other charges arrange themselves so that the field on the inside cancels, which doubles it on the outside.

**Worked example 9.** A point charge $q$ at the centre of a hollow conducting shell with inner radius $a$ and outer radius $b$, net charge zero. A Gaussian sphere inside the metal ($a < r < b$) must have zero flux, so $Q_{\text{enc}} = 0$: a charge $-q$ is induced on the inner surface. The shell is neutral, so $+q$ appears on the outer surface. The field is $q/4\pi\epsilon_0r^2$ for $r < a$, zero in the metal, and $q/4\pi\epsilon_0r^2$ again for $r > b$. The outside world cannot tell the shell is there. The code reproduces it with the inner charge plus two shells of induced charge.

In [ ]:
a_in, b_out = 0.5, 0.8
pos_in, q_in = shell_charges(a_in, -Q, 20000); pos_out, q_out = shell_charges(b_out, Q, 20000)
src = np.vstack([[[0, 0, 0]], pos_in, pos_out]); chg = np.concatenate([[Q], q_in, q_out])
r_line = np.array([0.2, 0.4, 0.55, 0.65, 0.75, 1.0, 2.0])
E9 = np.einsum("ij,j->i", fields.point_charges_E(src, chg, r_line[:, None] * direction), direction)
print("point charge inside a neutral conducting shell (inner radius 0.5, outer 0.8)")
for r, e in zip(r_line, E9):
    region = "cavity" if r < a_in else ("metal " if r < b_out else "outside")
    print(f"  r = {r:.2f} {region}:  E_r = {e:10.3f}    Gauss: {0.0 if a_in < r < b_out else K * Q / r**2:10.3f}")

### 3.8 The differential form

Primer 00c defined divergence as flux per unit volume. Apply Gauss' law to a tiny box: flux out $= \rho\,\Delta v/\epsilon_0$, and flux out divided by $\Delta v$ is the divergence. So

$$\nabla\cdot\mathbf{E} = \frac{\rho}{\epsilon_0}.$$

This is Gauss' law at a point. It says the electric field has sources exactly where there is charge and nowhere else. For a point charge, $\nabla\cdot\mathbf{E} = 0$ everywhere except at the charge (primer 00d, worked example 6), where it is infinite in such a way that the volume integral is $q/\epsilon_0$. For the uniformly charged ball, differentiate the hand result: inside, $E = Qr/4\pi\epsilon_0R^3$ and the spherical divergence is $\frac1{r^2}\frac{d}{dr}(r^2E) = \frac{3Q}{4\pi\epsilon_0R^3} = \rho/\epsilon_0$. Outside, $\frac1{r^2}\frac{d}{dr}(r^2\cdot Q/4\pi\epsilon_0r^2) = 0$. The differential form is satisfied by the field we found with the integral form, as it must be.

**Worked example 10.** A field is given as $\mathbf{E} = c\,(x, y, z)$ in some region. What charge density produces it? $\nabla\cdot\mathbf{E} = 3c$, so $\rho = 3c\epsilon_0$, uniform. Compare with worked example 3: inside the ball, $\mathbf{E} = \dfrac{Q}{4\pi\epsilon_0R^3}\mathbf{r}$, so $c = Q/4\pi\epsilon_0R^3$ and $\rho = 3Q/4\pi R^3$, the ball's density. Consistent.

In [ ]:
# Divergence of the superposition field of the ball, on a grid, compared with rho/eps0 inside and 0 outside.
# A coarse grid: 30k sources times 25^3 observation points is already 500 million distance evaluations.
xg, yg, zg, G = grids.grid3d((-1.6, 1.6), (-1.6, 1.6), (-1.6, 1.6), 25)
hg = grids.spacing(xg, yg, zg)
E_grid = fields.point_charges_E(cells, q_cell, G)
div = fields.divergence(E_grid, *hg)
rg = grids.magnitude(G)
rho = 3 * Q / (4 * np.pi * R**3)
print(f"rho / eps0 = {rho / EPS0:.2f}")
print(f"div E averaged over 0.3R < r < 0.7R: {div[(rg > 0.3) & (rg < 0.7)].mean():.2f}   (spread {div[(rg > 0.3) & (rg < 0.7)].std():.2f}, lattice graininess)")
print(f"div E averaged over 1.2R < r < 1.5R: {div[(rg > 1.2) & (rg < 1.5)].mean():.2f}")

### Exercise: the ball, piecewise

Replace the body of `E_shell_and_ball` so it returns the radial field of a uniformly charged ball of radius $R_1$ and charge $Q_1$ surrounded by a thin shell of radius $R_2 > R_1$ and charge $Q_2$, for an array of radii. Three regions. The check compares with superposition at $0.7R_1$ and outside the ball. Deeper inside, the lattice graininess seen in section 3.3 spoils the brute-force sum at the few-percent level, and within a couple of cells of the ball's surface the staircased boundary does the same.

In [ ]:
def E_shell_and_ball(r, R1, Q1, R2, Q2):
    """Radial field of a ball (R1, Q1) inside a thin shell (R2, Q2)."""
    inside_ball = K * Q1 * r / R1**3                                   # replace from here with your own piecewise expression
    between = K * Q1 / r**2
    outside = K * (Q1 + Q2) / r**2
    return np.where(r < R1, inside_ball, np.where(r < R2, between, outside))


R1, Q1, R2, Q2 = 1.0, 1e-9, 1.5, -3e-9
pos2, q2 = shell_charges(R2, Q2, 20000)
r_test = np.array([0.7, 1.2, 1.4, 1.8, 3.0])
E_sup = np.einsum("ij,j->i", fields.point_charges_E(np.vstack([cells, pos2]), np.concatenate([q_cell, q2]), r_test[:, None] * direction), direction)
E_hand = E_shell_and_ball(r_test, R1, Q1, R2, Q2)
print("r:            ", r_test); print("superposition:", E_sup); print("by hand:      ", E_hand)
assert np.allclose(E_sup, E_hand, rtol=0.03)

## 4. Verify

Gauss' law for the ball, with the brute-force field. Compute the flux of the superposition field through spheres of radius $0.5R$, $0.8R$, $R$, $1.5R$ and $2R$ by sampling the field on patches. Inside, the flux should grow as $r^3$ (the enclosed charge does); outside it should be constant at $Q/\epsilon_0$.

In [ ]:
def sphere_flux(E_func, r, N=4000):
    pos, _ = shell_charges(r, 1.0, N)
    normals = pos / r
    E = E_func(pos)
    return np.sum(np.einsum("ij,ij->i", E, normals)) * (4 * np.pi * r**2 / N)

E_func = lambda pts: fields.point_charges_E(cells, q_cell, pts)
print("  r/R    flux / (Q/eps0)    (r/R)^3 inside, 1 outside")
for rr in (0.5, 0.8, 1.0, 1.5, 2.0):
    f = sphere_flux(E_func, rr * R)
    print(f"  {rr:4.1f}   {f / (Q / EPS0):14.4f}   {min(rr, 1.0) ** 3:10.4f}")

## 5. Explore and rebuild

Explore. Gauss' law holds for a cube of charge too, but the symmetry method fails: the field of a uniformly charged cube is not constant on any simple surface. Build a cube from point charges and sample $|\mathbf{E}|$ on a concentric sphere just outside it. How much does $|\mathbf{E}|$ vary around the sphere? Now move the sphere out to five times the cube's size: the variation shrinks and the field approaches that of a point charge. Quantify how fast, and relate it to what you saw for the finite plate in section 3.6.

Rebuild. Close this notebook. On paper, write the three symmetry arguments (spherical, cylindrical, planar) and derive worked examples 3, 5 and 7 from Gauss' law. Then derive the field just outside a conductor. In a blank editor, build the ball from point charges and reproduce the plot of section 3.3.